<a href="https://colab.research.google.com/github/brandywii/dsa405-project/blob/main/notebooks/01_extraction_and_verification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
###############################################################################################################################################################
# Author: Brandy Wilkinson
#         DSA 405
#         Fall 2026
#
# Description:
#         This script `01_extraction_and_verification.ipynb` serves as the data ingestion, cleaning,
#         and validation pipeline that prepares both raw data sources for downstream analysis.
#         The notebook loads the local USDA Pesticide Data Program extract
#         (AnalyticalResults.csv, containing 1,048 positive glyphosate detections across
#         soybeans and corn) while simultaneously fetching and caching the 157 federal Maximum
#         Residue Limits (MRLs) directly from the live eCFR API (40 CFR 180.364). It then standardizes the datasets by
#         mapping USDA's two-letter commodity codes (SY and CO) to their exact legal eCFR descriptions (Soybean, seed and Corn, field, grain) and
#         executes automated, fail-loud verification checks to confirm row counts, data types, unit consistency (ppm) parts per million, and data integrity.
#         Finally, the notebook performs a validated many_to_one merge to attach each individual sample measurement to its corresponding EPA legal ceiling
#         (20 ppm for soybeans and 5.0 ppm for corn) and exports the clean, merged dataset for '02_analysis_and_figures.ipynb'
#
#
#
# Issues:
#        No issues detected
# ###############################################################################################################################################################

# load relevant packages
import io
import time
from pathlib import Path
import pandas as pd
import requests

# Define paths & github repo url
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CACHE_FILE = RAW_DIR / "usda_ecfr_cache.json"
PDP_COPY = RAW_DIR / "AnalyticalResults.csv"
ECFR_HTML_COPY = RAW_DIR / "ecfr_180_364.html"

# URL for each dataset
GITHUB_CSV_URL = "https://raw.githubusercontent.com/brandywii/dsa405-project/main/data/raw/AnalyticalResults.csv"
ECFR_URL = "https://www.ecfr.gov/api/renderer/v1/content/enhanced/2026-09-01/title-40?chapter=I&subchapter=E&part=180&subpart=C&section=180.364"

In [ ]:
def retrieve_validate_data():
    # Caching check -> Return cached files if exist locally
    if CACHE_FILE.exists() and PDP_COPY.exists() and ECFR_HTML_COPY.exists():
        print("Loading cached data from 'data/raw/'.")
        return (
            pd.read_csv(PDP_COPY),
            pd.read_json(CACHE_FILE),
            ECFR_HTML_COPY.read_text(encoding="utf-8")
        )

    print("Fetching dataset from brandywii/dsa405-project GitHub repo...")
    try:
        pdp_df = pd.read_csv(GITHUB_CSV_URL)
    except Exception as e:
        raise ConnectionError(f"FATAL: Failed to download dataset from GitHub: {e}")

    # Rate limited eCFR API request with fail-loud handling
    # honest user agent provided
    headers = {"User-Agent": "NCSU DSA405-student"}
    time.sleep(1.0)
    res = requests.get(ECFR_URL, headers=headers, timeout=30)

    if res.status_code != 200:
        raise RuntimeError(f"FATAL: eCFR API returned non-200 status code: {res.status_code}")

    # Persist raw files to local disk to satisfy caching rules
    status_df = pd.DataFrame([
        {"Source": "eCFR API", "Status": res.status_code},
        {"Source": "GitHub PDP", "URL": GITHUB_CSV_URL, "Status": 200}
    ])
    status_df.to_json(CACHE_FILE, orient="records")
    pdp_df.to_csv(PDP_COPY, index=False)
    ECFR_HTML_COPY.write_text(res.text, encoding="utf-8")
    print(f"Raw data successfully cached to {RAW_DIR}")

    return pdp_df, status_df, res.text

# Execute the acquisition pipeline
pdp_df, df_sources, html_data = retrieve_validate_data()
print(f"\nSuccessfully loaded PDP dataset with {len(pdp_df)} rows.")

Loading cached data from 'data/raw/'.

Successfully loaded PDP dataset with 1048 rows.


In [ ]:
# Put data into a table
data_tables = pd.read_html(io.StringIO(html_data))

print(f"Num of tables found: {len(data_tables)}")

# Tag the source tables before combining to flag GMO vs Conventional accurately
# Table 0 represents standard/conventional tolerances
df_conv = data_tables[0].copy()
df_conv["Is_GMO"] = False

# Table 1 represents specialized/GMO-associated traits
df_gmo = data_tables[1].copy()
df_gmo["Is_GMO"] = True

# Combine standard/conventional and GMO tables
df_ecfr = pd.concat([df_conv, df_gmo], ignore_index=True)

# Coalesce the two separate tolerance columns into a single unified one called `Tolerance_ppm`
df_ecfr["Tolerance_ppm"] = df_ecfr["Parts per million"].combine_first(
    df_ecfr["Parts per Million"]
)

# Drop the old split columns
df_ecfr = df_ecfr.drop(columns=["Parts per million", "Parts per Million"])

# Reorder columns so 'Is_GMO' is the third column (index 2)
df_ecfr = df_ecfr[["Commodity", "Tolerance_ppm", "Is_GMO"]]

# Strip any whitespace or hidden formatting from commodity text
df_ecfr["Commodity"] = df_ecfr["Commodity"].str.strip()

# Display eCFR table shape
print(f"eCFR table rows and columns: {df_ecfr.shape}")

# Display data types
print("___ Data Types ___")
print(df_ecfr.dtypes)

# Display sum of missing values in each column
print("___ Missing Values ___")
print(df_ecfr.isna().sum())

# Display number of unique values in each column
print("___Distinct Values ___")
print(df_ecfr.nunique())

# Display all summary stats
print("___ Summary Stats ___")
print(df_ecfr.describe(include="all"))

# Save full cleaned eCFR tolerances for Notebook 2
df_ecfr.to_csv(PROCESSED_DIR / "ecfr_tolerances.csv", index=False)

# Display as HTML grid
df_ecfr

Num of tables found: 2
eCFR table rows and columns: (158, 3)
___ Data Types ___
Commodity         object
Tolerance_ppm    float64
Is_GMO              bool
dtype: object
___ Missing Values ___
Commodity        0
Tolerance_ppm    0
Is_GMO           0
dtype: int64
___Distinct Values ___
Commodity        158
Tolerance_ppm     28
Is_GMO             2
dtype: int64
___ Summary Stats ___
       Commodity  Tolerance_ppm Is_GMO
count        158     158.000000    158
unique       158            NaN      2
top      Acerola            NaN  False
freq           1            NaN    141
mean         NaN      20.398734    NaN
std          NaN      61.618256    NaN
min          NaN       0.050000    NaN
25%          NaN       0.200000    NaN
50%          NaN       0.200000    NaN
75%          NaN       4.625000    NaN
max          NaN     400.000000    NaN


,Commodity,Tolerance_ppm,Is_GMO
0,Acerola,0.2,False
1,"Alfalfa, seed",0.5,False
2,"Almond, hulls",25.0,False
3,Aloe vera,0.5,False
4,Ambarella,0.2,False
...,...,...,...
153,"Sheep, meat byproducts",5.0,True
154,"Soybean, forage",100.0,True
155,"Soybean, hay",200.0,True
156,"Soybean, hulls",120.0,True


In [ ]:
# Map PDP Commod codes ('SY', 'CO') to eCFR Commodity names & add Pesticide Code 653-Glyphosate
# There are only 2 commodities that have established EPA MRL AND detectable glyphosate measurement in USDA PDP
commodity_to_code = {
    "Soybean, seed": "SY",
    "Corn, field, grain": "CO"
}

ecfr_tolerances = df_ecfr[df_ecfr["Commodity"].isin(commodity_to_code.keys())].copy()
ecfr_tolerances["Commod"] = ecfr_tolerances["Commodity"].map(commodity_to_code)
ecfr_tolerances["Pesticide Code"] = 653
ecfr_tolerances["EPA_Tolerance_Limit"] = pd.to_numeric(ecfr_tolerances["Tolerance_ppm"], errors="coerce")

# De-duplicate on join keys so the left join stays strictly many-to-one
ecfr_tolerances = (
    ecfr_tolerances.sort_values(by="Is_GMO", ascending=False)
    .drop_duplicates(subset=["Commod", "Pesticide Code"], keep="first")
    [["Commod", "Pesticide Code", "Commodity", "EPA_Tolerance_Limit", "Is_GMO"]]
)

# execute a many-to-one left Join on ['Commod', 'Pesticide Code']
join_keys = ['Commod', 'Pesticide Code']
merged_df = pd.merge(pdp_df, ecfr_tolerances, on=join_keys, how='left')

print(f"Merge completed! Final merged shape: {merged_df.shape}")
merged_df

Merge completed! Final merged shape: (1048, 19)


,Sample ID,Commod,Pesticide Code,Pesticide Name,Test Class,Concentration,LOD,pp_,Confirm 1,Confirm 2,Annotate,Quantitate,Mean,Extract,Determ,EPA Tolerance (ppm),Commodity,EPA_Tolerance_Limit,Is_GMO
0,MO1009070024SYCO1,SY,653,Glyphosate,I,3.615,0.25,M,LU,NaN,NaN,NaN,O,999,52,20,"Soybean, seed",20.0,True
1,MO1009070031SYCO1,SY,653,Glyphosate,I,0.516,0.25,M,LU,NaN,NaN,E,O,999,52,20,"Soybean, seed",20.0,True
2,MO1009090035SYCO1,SY,653,Glyphosate,I,0.907,0.25,M,LU,NaN,NaN,NaN,O,999,52,20,"Soybean, seed",20.0,True
3,MO1009150034SYCO1,SY,653,Glyphosate,I,0.918,0.25,M,LU,NaN,NaN,NaN,O,999,52,20,"Soybean, seed",20.0,True
4,MO1009160036SYCO1,SY,653,Glyphosate,I,0.586,0.25,M,LU,NaN,NaN,NaN,O,999,52,20,"Soybean, seed",20.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1043,MO2301140596SYUS2,SY,653,Glyphosate,I,0.161,0.10,M,LU,NaN,NaN,NaN,O,997,52,20,"Soybean, seed",20.0,True
1044,MO2301140603SYUS2,SY,653,Glyphosate,I,0.419,0.10,M,LU,NaN,NaN,NaN,O,997,52,20,"Soybean, seed",20.0,True
1045,MO2301150608SYUS2,SY,653,Glyphosate,I,0.460,0.10,M,LU,NaN,NaN,NaN,O,997,52,20,"Soybean, seed",20.0,True
1046,MO2301170611SYUS2,SY,653,Glyphosate,I,0.722,0.10,M,LU,NaN,NaN,NaN,O,997,52,20,"Soybean, seed",20.0,True


In [ ]:
# Post join data integrity check
def verify_merged_data(df, n_expected):

    # 1. Row Count Assertion
    # Shields against fan-out joins multiplying rows or unexpected data loss
    assert len(df) == n_expected, (
        f"FATAL: Expected {n_expected} rows after left join, detected {len(df)} rows."
    )

    # 2. Key Uniqueness Assertion
    # Shields against duplicated primary keys inflating sample measurements
    assert df['Sample ID'].is_unique if 'Sample ID' in df.columns else True, (
        "FATAL: Duplicate sample IDs detected in primary measurement table."
    )

    # 3. Value Range Assertion
    # Shields against unrealistic, negative tolerance limits from merge errors
    if 'EPA_Tolerance_Limit' in df.columns:
        invalid_tolerances = df[df['EPA_Tolerance_Limit'] < 0]
        assert len(invalid_tolerances) == 0, (
            f"FATAL: Found {len(invalid_tolerances)} rows with negative EPA tolerance limits."
        )

    # 4. Referential Integrity / Non-Null Primary Key Assertion
    # Shields against missing or corrupted join keys in the primary measurement table
    assert df['Commod'].notnull().all(), (
        "FATAL: Unexpected null values detected in primary join key ('Commod')."
    )

    # 5. Cardinality & Fan-Out Limit Assertion
    # Shields against an accidental many-to-many explosion increasing the row count
    assert len(df) <= n_expected, (
        f"FATAL: Row count expansion detected. Detected {len(df)} rows, max expected was {n_expected}."
    )

    print("All verification suite assertions passed successfully!")

# Run on clean data
verify_merged_data(merged_df, 1048)

#########################################################################################################################
# Demonstrate verification suite failing loudly
try:
    # mutate data by artificially inflating row count to show fan-out
    mutated_df = pd.concat([merged_df, merged_df.head(10)], ignore_index=True)

    print("Testing verification suite on mutated (bad) data...")
    verify_merged_data(mutated_df, 1048)

except AssertionError as e:
    print("\nDETECTED THE EXPECTED DEFECT SUCCESSFULLY:")
    print(e)

# Export verified dataframe for Notebook 2
# click link to download .csv file

merged_df.to_csv(PROCESSED_DIR / "merged_glyphosate_data.csv", index=False)
print("\nExported verified dataset to ../data/processed/merged_glyphosate_data.csv")

All verification suite assertions passed successfully!
Testing verification suite on mutated (bad) data...

DETECTED THE EXPECTED DEFECT SUCCESSFULLY:
FATAL: Expected 1048 rows after left join, detected 1058 rows.

Exported verified dataset to ../data/processed/merged_glyphosate_data.csv
